# 05 Site explorer and the reactive sign convention

Two jobs:

1. **Unpack individual sites** — daily operational plots in the style of
   `dnsp_analysis/notebooks/02c_explore_voltvar_voltwatt`, plus per-site Volt-VAr
   curves against the AS/NZS 4777.2 requirement.
2. **Settle the reactive sign**, fleet-wide, with every step showing whether the number
   on screen is the **raw delivered** value or the **flipped stored** one.

## Sign provenance — read this before any plot

Two orientations exist, and every function here carries both:

| column | meaning |
|---|---|
| `Q_raw_var` | straight from the delivered Parquet, **nothing applied** |
| `Q_raw_kvar` | `Q_raw_var / 1000` — sign still untouched |
| `Q_stored_kvar` | what `se_interval` holds, and what D9/D10 scored: `-1 × raw / 1000` |

Active power is **not** transformed. `ACTIVE_POWER_SIGN = +1`, SolarEdge reports a
production magnitude, and there are no negative values anywhere in the delivery.

Every figure stamps its orientation in the title. If it says `STORED`, the flip is
applied. If it says `RAW`, it is as SolarEdge delivered it.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

_current = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_current, *_current.parents)
     if (p / "oem_analysis").is_dir() and (p / "bms_sa_review").is_dir()),
    None,
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from oem_analysis.config import se_config as C
from oem_analysis.lib import se_store, se_explore as ex, se_sign as sign
from oem_analysis.lib import se_params
from oem_analysis.lib import se_counterfactual as ctf

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

con = se_store.connect()
config = se_params.CONFIG    # same cohort/params as notebooks 02-03
print(f"REACTIVE_POWER_SIGN = {C.REACTIVE_POWER_SIGN:+.0f}   "
      f"(stored Q = {C.REACTIVE_POWER_SIGN:+.0f} x raw / 1000)")
print(f"ACTIVE_POWER_SIGN   = {C.ACTIVE_POWER_SIGN:+.0f}   (no change)")

## 1. Verify the transform against the raw files

Before arguing about which orientation is correct, confirm what the store actually did.

This re-reads the delivered Parquet for one site, re-applies the timezone conversion
independently, joins on the UTC instant, and checks that
`Q_stored_kvar == -1 × Q_raw_var / 1000` on every row. It also checks active power was
left alone.

If `n_Q_mismatched` is anything but 0, stop — the store is not what the rest of this
notebook assumes.

In [ ]:
SITE = "AUS068"        
# SITE = "AUS351"
display(ex.verify_sign_transform(con, SITE).T)
display(ex.site_profile(con, SITE))

## 2. The per-site curve — which orientation matches the standard?

Median Q against voltage for one site, drawn **both ways**, with the required curve and
its ±4% tolerance band in magenta.

The question the plot asks: *which trace sits on the same side of zero as the
requirement, and inside its tolerance band?*

In [ ]:
curve = ex.site_voltvar_curve(con, SITE)
display(ex.plot_site_voltvar_curve(curve, SITE, ex.site_profile(con, SITE)))
display(curve.head(12));

## 3. The daily operational plot

`site_days_available()` ranks days by how much time they spend above 240 V, because a day
that never enters the Volt-VAr band shows a flat reactive trace and settles nothing.

Both orientations are plotted below. **Compare the fourth panel**: the blue measured
trace against the magenta required band.

In [ ]:
days = ex.site_days_available(con, SITE)
display(days.head(8))
days

In [ ]:
ZOOM_DATE = str(days.day_aest.iloc[0])[:10]
# ZOOM_DATE = "2025-04-16"     # override with any date from days.day_aest
# ZOOM_DATE = "2025-01-12"

df_day = ex.site_day(con, SITE, ZOOM_DATE)
assert len(df_day) > 0, f"No rows for {ZOOM_DATE}"
print(f"{len(df_day)} intervals | raw values joined: {df_day.Q_raw_var.notna().sum()}")
display(df_day[["ts_aest", "V", "P_kW", "Q_raw_var", "Q_raw_kvar", "Q_stored_kvar"]].iloc[::24])

In [ ]:
S = float(df_day.s_99.iloc[0])
display(ex.plot_operational(df_day, SITE, S, ZOOM_DATE, orientation="stored"))

### 3b. Volt-Watt — why the operational plot cannot answer this

`plot_operational` is the right tool for Volt-VAr and the wrong one for Volt-Watt, and
that is not a defect in the figure.

**Volt-VAr** asks *is Q on the curve?* — answerable from Q and V alone, both measured.
**Volt-Watt** asks *is P below the ceiling?* — and a site sitting far below the ceiling
looks identical whether it curtailed correctly or simply had no sun. You cannot tell
those apart from measured data. It needs a third trace: what the site **could** have
produced.

That is why a site can be scored "Volt-Watt conformant" and still look like nothing is
happening. Most of the time nothing *is* — 71% of this fleet never reaches 253 V at all,
and among those that do, most exposed intervals sit well under the ceiling because the
sun was not there.

The plot below adds the ceiling and the counterfactual, and shades the two outcomes:

| shading | meaning |
|---|---|
| **orange** | power was available *and* the site held below the ceiling — Volt-Watt **working**, observed |
| **red** | measured P above the ceiling — **non-conformant** |

Neither shading, with P well under the ceiling, means the inverter was never asked to
respond. Conformant, but demonstrating nothing.

`uncurtailed_P` needs `se_uncurtailedpv` from notebook 04 §5. Without it the plot still
shows the ceiling and any breaches, but curtailment cannot be assessed — and it says so
in the title rather than leaving you to infer it.

In [ ]:
vw_day = ex.site_voltwatt_day(con, SITE, ZOOM_DATE, config)
display(ex.plot_voltwatt_day(vw_day, SITE, ZOOM_DATE, config))

# The intervals the verdict actually rests on
exposed = vw_day[vw_day.exposed.fillna(False)]
print(f"{len(exposed):,} of {len(vw_day):,} intervals exposed (V > 253 V)")
if len(exposed):
    display(exposed[["ts_aest", "V", "P_kW", "P_ceiling_kW", "uncurtailed_P",
                     "nonconformant", "curtailed"]].head(12))

#### Pick a day that actually tests Volt-Watt

A day with no exposure tells you nothing about the response. This ranks the site's days
by how much high voltage they saw, so the plot above can be pointed at a day where the
question was genuinely asked.

In [ ]:
vw_days = con.execute(f"""
    SELECT CAST(ts_aest AS DATE)                            AS day_aest,
           count(*)                                          AS n_intervals,
           count(*) FILTER (WHERE V_mean > {C.as4777()['VW']['V1']}) AS n_above_253,
           round(max(V_mean), 1)                             AS v_max,
           round(max(P_kW), 2)                               AS p_max_kW
    FROM se_interval
    WHERE site_alias = '{SITE}'
    GROUP BY 1
    HAVING n_above_253 > 0
    ORDER BY n_above_253 DESC
    LIMIT 15
""").df()
display(vw_days)

if len(vw_days):
    VW_DATE = str(vw_days.day_aest.iloc[0])[:10]
    print(f"Most-exposed day: {VW_DATE}")
    display(ex.plot_voltwatt_day(
        ex.site_voltwatt_day(con, SITE, VW_DATE, config), SITE, VW_DATE, config))
else:
    print(f"{SITE} never exceeded 253 V — Volt-Watt was never tested for this site.")

#### Per-timestamp classification — the Volt-Watt analogue of the Volt-VAr table

Five categories, not two, because **"P below the ceiling" covers three genuinely
different situations** and collapsing them is what makes a "conformant" verdict hard to
read:

| category | meaning | evidence value |
|---|---|---|
| `not_exposed` | V ≤ 253 V | no limit applied |
| `nonconformant` | P above the ceiling | **proof it failed** |
| `curtailed` | below ceiling, counterfactual says the sun *was* there | **proof it works** |
| `sun_limited` | below ceiling, counterfactual says the sun was *not* there | nothing tested |
| `no_counterfactual` | below ceiling, no prediction available | unanswerable |

Only two of the five carry evidence. `uncurtailed_P` is on every row, so you can see the
number the classification turned on.

In [ ]:
vw_cats = ex.site_voltwatt_categories(con, SITE, VW_DATE, config)
display(ex.voltwatt_day_summary(vw_cats))

# Every exposed interval, with the counterfactual that decided its category
display(vw_cats[vw_cats.category != "not_exposed"].head(30))

vw_cats.to_csv(C.ARTEFACT_DIR / f"voltwatt_intervals_{SITE}_{VW_DATE}.csv", index=False)
print(f"-> {C.ARTEFACT_DIR / f'voltwatt_intervals_{SITE}_{VW_DATE}.csv'}")

#### Why is the counterfactual patchy?

Because it is produced **per five-minute time-of-day bin**, not per interval. Each bin
gets its own regression, and a bin needs at least 5 surviving training rows to get one at
all. If a site's 11:25 bin never accumulated five usable rows across the whole record,
then *every* 11:25 in the year has no prediction — which is why the dashed line is
speckled rather than simply truncated.

So yes, this is expected behaviour. But it is a **coverage limitation to quantify, not
decoration** — every gap is an interval that falls into the `uncurtailed_P IS NULL`
fallback in notebook 03 §5b, where it gets scored by the basic test instead.

The two cells below say *which* filter is responsible for this particular site. Do not
assume — the binding constraint differs between sites, and on the site I profiled the
biggest cut was **not** the one you would guess.

In [ ]:
display(ex.counterfactual_training_attrition(con, SITE))

In [ ]:
gaps = ex.voltwatt_counterfactual_gaps(con, SITE, VW_DATE, config)
gaps["hour"] = pd.to_datetime(gaps.tod_bin.astype(str), format="%H:%M:%S").dt.hour

by_hour = gaps.groupby("hour").agg(
    bins=("tod_bin", "size"),
    bins_with_model=("model_exists", "sum"),
    mean_V=("mean_V", "mean"),
    exposed_intervals=("n_exposed", "sum"),
).round(1)
by_hour["pct_covered"] = (100 * by_hour.bins_with_model / by_hour.bins).round(0)
display(by_hour)

print(f"{int(gaps.model_exists.sum())} of {len(gaps)} time-of-day bins on this day "
      f"have a model.")
print("Coverage improves with record length — a 3-month store is far patchier than 12.")

#### Which step lost each missing interval

A row must survive **five** things to get a counterfactual:

| # | step | fails when |
|---|---|---|
| 1 | reach `se_structured` | no BOM irradiance for that (postcode, 5-min slot) |
| 2 | `GHI_cs > 0` | no clear-sky irradiance reference at that time of day |
| 3 | `P_kw_norm_cs > 0` | no clear-sky power reference at that time of day |
| 4 | a model for `(site, tod_bin)` | the bin had too few training rows |
| 5 | site passed the MAPE gate | model can't predict this site at all (≤ 50% median error) |

Steps 4 and 5 are properties of the **site and clock time** — if they're the cause, that
same clock time is missing on *every* day of the record. Step 1 is a property of **that
day** — a gap in the satellite feed. On a single day plot the two look identical and mean
completely different things, so `bin_covered_other_days` separates them:

- `= 0` → **structural**: model or gate. Missing all year.
- `> 0` → **transient**: BOM gap on this day only.

**A telltale for step 1:** BOM is 10-minute data duplicated into 5-minute slots (`t` and
`t+5`), so **one** missing satellite record removes **exactly two adjacent** 5-minute
slots. Two consecutive gaps in the middle of an otherwise covered block is that signature
— not a training-data problem.

In [ ]:
missing = ctf.explain_missing_counterfactual(con, SITE, VW_DATE, config)
if len(missing):
    display(missing[["ts_aest", "tod_bin", "in_structured", "GHI_cs",
                     "P_kw_norm_cs", "model_exists", "model_train_points",
                     "bin_covered_other_days", "reason", "verdict"]])

### 3c. Volt-VAr scatter — *where on the curve* the site fails

The daily plot shows **when** a site misbehaves. This shows **where on the curve**, and
the two failure shapes look completely different:

- a **flat horizontal band near Q = 0** across the whole voltage range → Volt-VAr is
  **disabled**. The inverter never responds at any voltage.
- points that **track the required curve but sit short of it** → responding with the
  **wrong settings**, or a smaller capability than assumed.

Both produce similar non-conformance percentages. Only this view separates them, and they
call for different follow-up — one is a commissioning problem, the other a settings
problem.

Colours come from `site_interval_categories`, which runs the **same D9 SQL that produced
the fleet numbers**, so a point coloured "adverse" here is one of the intervals counted as
adverse in notebook 03. The explorer and the headline cannot drift apart.

In [ ]:
# Whole store for this site; pass a slice for a single month.
cats_all = ex.site_interval_categories(con, SITE, config=config)
profile = ex.site_profile(con, SITE, config)

display(ex.plot_voltvar_scatter(
    cats_all, SITE, float(profile.s_99),
    period_label=f"{cats_all.ts_aest.min():%Y-%m-%d} to {cats_all.ts_aest.max():%Y-%m-%d}",
    config=config))

One month at a time, if a site's behaviour changed part-way through the year — a firmware
update or a re-commissioning shows up as the cloud moving between months.

In [ ]:
MONTH = str(cats_all.ts_aest.dt.to_period("M").iloc[-1])   # e.g. "2025-12"
month_cats = cats_all[cats_all.ts_aest.dt.to_period("M").astype(str) == MONTH]

if len(month_cats):
    display(ex.plot_voltvar_scatter(month_cats, SITE, float(profile.s_99),
                                    period_label=MONTH, config=config))
else:
    print(f"No intervals for {SITE} in {MONTH}.")

In [ ]:
cats = ex.site_interval_categories(con, SITE, ZOOM_DATE)
display(cats.category.value_counts().rename("intervals").to_frame())
display(cats[["ts_aest", "V", "P_kW", "Q_kvar", "Q_min_permitted", "Q_max_permitted",
              "Q_impact", "category", "dist_to_band_kvar"]])

## 4. Fleet-wide: which orientation fits the required curve?

Not a sample — **every site in the fleet**.

For each site, across the Volt-VAr ramp only (241–253 V, where the requirement is
non-zero and Volt-Watt has not yet engaged), compute the median absolute deviation of
measured Q from the required curve in both orientations, and compare against the ±4%
tolerance band.

Three outcomes:

- **`raw fits`** — as-delivered is within tolerance, flipped is not. The flip broke it.
- **`stored fits`** — the reverse. The flip was right for this site.
- **`neither fits`** — the site is not following the curve either way, so it carries no
  information about the convention and should be excluded from the argument.

In [ ]:
'''
fit = sign.fleet_orientation_fit(con)
print(f"{len(fit):,} of 1,602 sites had enough data in the ramp\n")
display(pd.crosstab(fit.cohort, fit.verdict, margins=True))

informative = fit[fit.verdict.str.contains(r"fits \(within")]
print(f"\nInformative sites (fit exactly one orientation): {len(informative):,}")
display(pd.crosstab(informative.cohort, informative.verdict))

fit.to_csv(C.ARTEFACT_DIR / "fleet_orientation_fit.csv", index=False)
print(f"\nFull per-site table -> {C.ARTEFACT_DIR / 'fleet_orientation_fit.csv'}")
'''

## 5. Look at any site

Change `SITE` and re-run. Useful starting points from the fleet table above:

```python
fit[fit.verdict == "raw fits (within tol)"].head(20)      # flip broke these
fit[fit.verdict == "stored fits (within tol)"].head(20)   # flip was right for these
fit.nsmallest(20, "mad_raw_kvar")                          # cleanest curve-followers
```

In [ ]:
def explore(site, day=None, orientation="raw"):
    """Profile + curve + daily plot for one site. Returns the day frame."""
    display(ex.site_profile(con, site))
    display(ex.plot_site_voltvar_curve(ex.site_voltvar_curve(con, site), site,
                                       ex.site_profile(con, site)))
    avail = ex.site_days_available(con, site)
    day = day or str(avail.day_aest.iloc[0])[:10]
    frame = ex.site_day(con, site, day)
    display(ex.plot_operational(frame, site, float(frame.s_99.iloc[0]), day,
                                orientation=orientation))
    return frame

# The fleet-wide orientation fit in the cell above is commented out (it is a
# whole-fleet scan). Pick a site directly rather than indexing into `fit`, which
# that cell would have defined — otherwise this raises NameError on a clean run.
_ = explore(SITE, orientation="stored")

# With the fit cell uncommented, this selects a site that fits the stored sign:
#     _ = explore(fit[fit.verdict == "stored fits (within tol)"].site_alias.iloc[0],
#                 orientation="stored")

### Per-timestamp categorisation

The same D9 interval scoring, restricted to one site, so the table and the plot are
guaranteed to describe the same thing.

| category | meaning |
|---|---|
| `not_assessable` | \|P\| < 20% of `s_99` — the standard sets no requirement |
| `within_band` | assessable and inside the permitted band — **this is conformance** |
| `Q_adverse` … `Q_major_surplus` | assessable and *outside* the band, split by `Q_impact` |

The five `Q_*` buckets only fire when Q is outside the band, so `within_band` and
`not_assessable` are what account for everything else. A day where the reactive trace
sits inside the amber band all afternoon should be almost entirely those two.

In [ ]:
cats = ex.site_interval_categories(con, SITE, ZOOM_DATE)
display(cats.category.value_counts().rename("intervals").to_frame())
display(cats[["ts_aest", "V", "P_kW", "Q_kvar", "Q_min_permitted", "Q_max_permitted",
              "Q_impact", "category", "dist_to_band_kvar"]])

## 4. The whole year in one figure

A single day proves nothing about an annual rate. This shows every day of the site's
year: **dates across, categories down, colour = % of that day's intervals.**

**Why it is fast.** The aggregation to (day × category) happens in DuckDB, so this
returns at most 365 × 7 = 2,555 cells rather than ~100,000 intervals. It renders in
well under a second. Do *not* fetch the intervals and pivot them client-side.

**How to read it.** A site that is genuinely fine shows a solid `within_band` row with
everything below it empty. Non-conformance appearing as a **vertical stripe** is
episodic — a handful of bad days. As a **horizontal band** it is persistent behaviour.
An annual percentage cannot tell those apart, which is exactly why this is worth
plotting before trusting a site-level verdict.

In [ ]:
calendar = ex.site_category_calendar(con, SITE)
display(ex.plot_category_heatmap(calendar, SITE))

reduced = calendar[["Q_adverse", "Q_inactive", "Q_significant_shortfall"]].sum(axis=1)
print(f"Days with ANY reduced non-conformance: {int((reduced > 0).sum())} of {len(calendar)}")
print(f"Worst days:")
display(calendar.assign(reduced_nonconf_pct=reduced.round(2))
        .nlargest(5, "reduced_nonconf_pct")[["reduced_nonconf_pct", "within_band",
                                             "not_assessable"]].round(1))

## What this establishes

The reactive sign is **inconsistent across the fleet**, and it does not split by phase
count. Within single-phase alone the informative sites divide almost evenly between the
two orientations.

That means **no global constant is correct** — neither the flip currently applied nor
leaving the raw value alone. `se_config.REACTIVE_POWER_SIGN` cannot express this.

It also cannot be fixed by inference. Deriving each site's orientation from whether it
absorbs above 240 V assumes the very behaviour a conformance study is measuring, and
would produce a 100% direction-conformance result by construction.

So until SolarEdge confirms the convention, or a site with known ground truth is
available:

- **magnitude conformance is assessable** — how much reactive power a site delivers
  relative to the requirement is orientation-independent;
- **direction conformance is not** — `Q_adverse` cannot be distinguished from a
  reporting-polarity difference.

The D9 Volt-VAr category results should be read with that restriction, for **both**
cohorts, not just three-phase.